# ORTHRUS-MSTC-PIDS — C8.2 Production Colab Notebook

This notebook is the production entry point for bounded-memory, restartable preprocessing and experiments. The paper configuration fits Word2Vec with `corpus_scope=train_only`; `official_full_dataset` is available only for original-ORTHRUS compatibility.

Runtime persistence model:

- The PostgreSQL server, apt packages, and `/content/orthrus` clone live in the temporary Colab VM and disappear after Runtime reset.
- Database dumps and preprocessing artifacts live in Google Drive and persist.
- Completion markers plus key-file validation allow each preprocessing stage to resume safely.
- `build_graphs` is primarily CPU RAM + PostgreSQL + Drive I/O; it does not require a GPU.
- Do not use “Run all”. Review the parameter cell, run setup cells in order, then explicitly enable one heavy stage.
- No training, preprocessing, restore, benchmark, or experiment matrix is enabled by default.


## 0. Unified parameters

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/orthrus")
DRIVE_ROOT = Path("/content/drive/MyDrive/mstc_pids")
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
DATA_ROOT = DRIVE_ROOT / "data"

REPOSITORY_URL = "https://github.com/fish23611-beep/orthrus.git"
REPOSITORY_REF = "fix/c8-bounded-baseline-smoke"
# Pin to the first-stage commit (production fix + tests only).
# The notebook itself is committed in a separate second-stage commit.
EXPECTED_COMMIT = "9cd53cd5d9aaf047bdaf50597f34d5072b414b83"

DATASET = "THEIA_E3"
SEEDS = [0]
CORPUS_SCOPE = "train_only"  # paper: train_only; compatibility: official_full_dataset

DB_DUMPS = {
    "THEIA_E3": DATA_ROOT / "database_dumps/theia_e3.dump",
    "THEIA_E5": DATA_ROOT / "database_dumps/theia_e5.dump",
}
DATABASE_DUMP = DB_DUMPS[DATASET]

MOUNT_DRIVE = True
CLONE_OR_UPDATE = True
INSTALL_DEPENDENCIES = True
RESTORE_DATABASE = False
FORCE_DATABASE_RESTORE = False
RUN_PREPROCESS_BENCHMARK = False
RUN_PREPROCESS = False
FORCE_PREPROCESS = False
PREPROCESS_SUBSTAGES = "build_graphs,embed_nodes,embed_edges"
RUN_BASELINE_SMOKE = False
SMOKE_MAX_WINDOWS_PER_SPLIT = 2  # smoke: 2 windows per split; None = full data
RUN_MAIN_MATRIX = False
RUN_ABLATIONS = False
RUN_COLLECT_EXPORT = False
RUN_DISPLAY_RESULTS = False
RUN_MANUAL_RESUME = False

EXPERIMENT_GROUP = "ablation"
RESUME_CONFIG = PROJECT_ROOT / "config/experiments/mstc_full.yml"
CHECKPOINT = Path("")

assert DATASET in DB_DUMPS
assert CORPUS_SCOPE in {"train_only", "official_full_dataset"}
print(f"Dataset={DATASET}; corpus_scope={CORPUS_SCOPE}; ref={REPOSITORY_REF}")
print(f"Pinned commit: {EXPECTED_COMMIT}")
print("Heavy-stage switches are all disabled by default.")


## 1. Mount Drive and configure persistent roots

In [ ]:
import os

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        print("Not running in Colab; Drive mount skipped.")

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
(DATA_ROOT / "database_dumps").mkdir(parents=True, exist_ok=True)
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)
print(f"Persistent artifact root: {ARTIFACT_ROOT}")


## 2. Checkout one coherent repository ref

In [ ]:
import subprocess
import sys


def git(*args, capture=False, check=True):
    return subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), *args],
        check=check, text=True, capture_output=capture,
    )


# -------------------------------------------------------------------------
# Strict checkout: verify remote branch, pin to CODE_SHA, detached HEAD
# -------------------------------------------------------------------------
if CLONE_OR_UPDATE:
    if (PROJECT_ROOT / ".git").is_dir():
        if git("status", "--porcelain", capture=True).stdout.strip():
            raise RuntimeError("Existing Colab clone is dirty; resolve it before checkout.")
    elif PROJECT_ROOT.exists() and any(PROJECT_ROOT.iterdir()):
        raise RuntimeError(f"Non-Git project directory is not empty: {PROJECT_ROOT}")
    else:
        PROJECT_ROOT.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

    # Fetch all refs
    git("fetch", "origin", "--tags", "--prune")

    # Verify remote branch exists
    remote_ref = f"origin/{REPOSITORY_REF}"
    if not git("show-ref", "--verify", f"refs/remotes/{remote_ref}", capture=True, check=False).returncode == 0:
        raise RuntimeError(f"Remote branch {remote_ref} does not exist. Available branches:")
    branches = git("branch", "-r", capture=True, check=False).stdout
    print(f"Available origin/* branches (excerpt): {branches[:500]}")

    # Verify EXPECTED_COMMIT exists
    if not git("cat-file", "-e", f"{EXPECTED_COMMIT}^{{commit}}", capture=True, check=False).returncode == 0:
        raise RuntimeError(f"EXPECTED_COMMIT {EXPECTED_COMMIT} does not exist in repository.")

    # Verify EXPECTED_COMMIT is an ancestor of the remote branch (ensures code is on branch)
    git("fetch", "origin", REPOSITORY_REF, "--tags")
    merge_base = git("merge-base", EXPECTED_COMMIT, f"origin/{REPOSITORY_REF}", capture=True, check=False).stdout.strip()
    if merge_base != EXPECTED_COMMIT:
        raise RuntimeError(
            f"EXPECTED_COMMIT {EXPECTED_COMMIT} is NOT an ancestor of origin/{REPOSITORY_REF}. "
            f"Merge-base={merge_base}. Pin to the correct production fix commit."
        )

    # Detached HEAD at pinned commit
    git("checkout", "--detach", EXPECTED_COMMIT)

# Post-checkout verification
actual_commit = git("rev-parse", "HEAD", capture=True).stdout.strip()
if actual_commit != EXPECTED_COMMIT:
    raise RuntimeError(f"Commit mismatch: expected {EXPECTED_COMMIT}, actual {actual_commit}")

working_tree = git("status", "--porcelain", capture=True).stdout
if working_tree.strip():
    raise RuntimeError(f"Working tree is not clean after checkout: {working_tree}")

print(f"Repository ref:   {REPOSITORY_REF}")
print(f"Pinned commit:    {EXPECTED_COMMIT}")
print(f"Actual commit:    {actual_commit}")
print("Clean:            True")

# Add src to sys.path
src_root = str(PROJECT_ROOT / "src")
if src_root not in sys.path:
    sys.path.insert(0, src_root)
print(f"sys.path updated: {src_root}")


## 3. Resource preflight

In [ ]:
import shutil
import subprocess
import sys
import psutil
import torch

print(f"Python: {sys.version.split()[0]}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=True)
ram = psutil.virtual_memory()
print(f"CPU RAM: total={ram.total/1024**3:.1f} GB, available={ram.available/1024**3:.1f} GB")
for disk_path in (Path("/content"), DRIVE_ROOT):
    if disk_path.exists():
        usage = shutil.disk_usage(disk_path)
        print(f"{disk_path}: total={usage.total/1024**3:.1f} GB, free={usage.free/1024**3:.1f} GB")

# Preprocessing (build_graphs / metadata / Word2Vec / embed_edges) runs on CPU.
# Only baseline smoke, main matrix, and ablation experiments require GPU.
print("Preprocessing (build_graphs, embed_nodes, embed_edges) runs on CPU.")
print("Baseline smoke / main matrix / ablations require GPU.")
GPU_READY = torch.cuda.is_available()


## 4. Install Python dependencies

In [ ]:
import subprocess
import sys

if INSTALL_DEPENDENCIES:
    packages = [
        "scikit-learn", "networkx", "xxhash", "graphviz", "psutil",
        "matplotlib", "wandb", "chardet", "nltk", "igraph", "cairocffi",
        "wget", "gensim", "pytz", "pandas", "yacs", "psycopg2-binary",
        "tqdm", "pyyaml", "torch_geometric",
    ]
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)
    torch_base = torch.__version__.split("+")[0]
    cuda_tag = "cpu" if torch.version.cuda is None else "cu" + torch.version.cuda.replace(".", "")
    wheel_index = f"https://data.pyg.org/whl/torch-{torch_base}+{cuda_tag}.html"
    for package in ("pyg_lib", "torch_scatter", "torch_sparse"):
        if subprocess.run([sys.executable, "-m", "pip", "show", package], capture_output=True).returncode:
            subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package, "-f", wheel_index], check=True)


## 5. Import smoke test and environment record

In [ ]:
import importlib
import subprocess
import sys
import tempfile
import torch
import networkx as nx
from torch_geometric.data import TemporalData

# Core imports
for module_name in ("config", "orthrus", "torch", "torch_geometric", "pandas", "yaml", "gensim", "nltk"):
    module = importlib.import_module(module_name)
    print(f"import {module_name}: ok {getattr(module, '__version__', '')}")

# NLTK smoke test + provnet_utils tokenization
from provnet_utils import (
    _ensure_nltk_punkt,
    tokenize_file,
    tokenize_subject,
    tokenize_netflow,
)

# Ensure punkt / punkt_tab are available
_ensure_nltk_punkt()
print("NLTK punkt/punkt_tab smoke test passed.")

# Minimal tokenization sanity checks
test_file = tokenize_file("/home/admin/profile/test.txt")
test_subj = tokenize_subject("/usr/bin/bash -c echo hello")
test_netflow = tokenize_netflow("128.55.12.110:443")

assert test_file, "tokenize_file returned empty"
assert test_subj, "tokenize_subject returned empty"
assert test_netflow, "tokenize_netflow returned empty"
print(f"tokenize_file:   {test_file[:3]}...")
print(f"tokenize_subject: {test_subj[:3]}...")
print(f"tokenize_netflow: {test_netflow}")

# -------------------------------------------------------------------------
# PyTorch 2.6 trusted-artifact serialization smoke test
# PyTorch 2.6 changed torch.load() default from weights_only=False to
# weights_only=True. ORTHRUS preprocessing generates full Python objects
# (NetworkX MultiDiGraph, PyG TemporalData) that require weights_only=False.
# This smoke catches serialization incompatibility before processing 194 graphs.
# -------------------------------------------------------------------------
from serialization_compat import load_trusted_torch_artifact

with tempfile.TemporaryDirectory() as tmpdir:
    # Smoke A: NetworkX MultiDiGraph roundtrip
    graph = nx.MultiDiGraph()
    graph.add_node(0, node_type="subject")
    graph.add_node(1, node_type="file")
    graph.add_edge(0, 1, label="write", time=1234567890)
    nx_path = f"{tmpdir}/smoke_graph.pkl"
    torch.save(graph, nx_path)
    loaded_graph = load_trusted_torch_artifact(nx_path, expected_type=nx.MultiDiGraph)
    assert isinstance(loaded_graph, nx.MultiDiGraph)
    assert loaded_graph.number_of_nodes() == 2
    assert loaded_graph.number_of_edges() == 1
    print("Smoke A (NetworkX MultiDiGraph): ok")

    # Smoke B: PyG TemporalData roundtrip
    td = TemporalData()
    td.src = torch.tensor([0, 1], dtype=torch.long)
    td.dst = torch.tensor([1, 0], dtype=torch.long)
    td.t = torch.tensor([1000, 1001], dtype=torch.long)
    td.msg = torch.randn(2, 16, dtype=torch.float32)
    td_path = f"{tmpdir}/smoke_window.TemporalData.simple"
    torch.save(td, td_path)
    loaded_td = load_trusted_torch_artifact(td_path, expected_type=TemporalData)
    assert isinstance(loaded_td, TemporalData)
    assert torch.equal(loaded_td.src, td.src)
    assert torch.equal(loaded_td.msg, td.msg)
    print("Smoke B (PyG TemporalData): ok")

print("PyTorch trusted-artifact serialization smoke test passed.")

# Persist pip freeze
environment_dir = ARTIFACT_ROOT / "environment"
environment_dir.mkdir(parents=True, exist_ok=True)
with (environment_dir / "pip_freeze.txt").open("w", encoding="utf-8") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True, text=True)
print(f"Environment recorded: {environment_dir / 'pip_freeze.txt'}")


## 6. Resolve configuration and inspect persistent artifacts

In [ ]:
from config import get_runtime_required_args, get_yml_cfg
from pipeline_stages import check_all_preprocess_stages_complete, check_preprocess_stage_complete

PREPROCESS_CONFIG = PROJECT_ROOT / "config/orthrus.yml"


def fresh_preprocess_cfg():
    args = get_runtime_required_args(args=[
        DATASET, "--config", str(PREPROCESS_CONFIG),
        "--artifact-root", str(ARTIFACT_ROOT), "--stages", "preprocess",
        f"--semantic_features.corpus_scope={CORPUS_SCOPE}", "--skip-tracing",
    ])
    return get_yml_cfg(args)


def artifact_stats(path):
    path = Path(path)
    files = [item for item in path.rglob("*") if item.is_file()] if path.is_dir() else []
    total = sum(item.stat().st_size for item in files)
    return {"files": len(files), "MB": total / 1024**2, "GB": total / 1024**3}


def read_preprocess_status(current_cfg):
    status = {
        stage: check_preprocess_stage_complete(current_cfg, stage)
        for stage in ("build_graphs", "embed_nodes", "embed_edges", "metadata")
    }
    status["artifacts_complete"] = check_all_preprocess_stages_complete(current_cfg)
    return status


def print_status(label, status):
    print(label)
    for stage in ("build_graphs", "embed_nodes", "embed_edges", "metadata"):
        print(f"  {stage}: {'✓' if status[stage] else '✗'}")
    print(f"  artifacts_complete: {status['artifacts_complete']}")

cfg = fresh_preprocess_cfg()
status_before = read_preprocess_status(cfg)
print_status("Current:", status_before)

artifact_paths = {
    "build_graphs": cfg.graph_construction.build_graphs._graphs_dir,
    "word2vec": cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir,
    "edge_embeddings": cfg.edge_featurization.embed_edges._edge_embeds_dir,
    "metadata": cfg._metadata_dir,
    "whole_ARTIFACT_ROOT": ARTIFACT_ROOT,
}
for label, artifact_path in artifact_paths.items():
    stats = artifact_stats(artifact_path)
    print(f"{label}: files={stats['files']}, MB={stats['MB']:.2f}, GB={stats['GB']:.3f}")

# Status-based decision logic
build_graphs = status_before["build_graphs"]
metadata     = status_before["metadata"]
embed_nodes  = status_before["embed_nodes"]
embed_edges  = status_before["embed_edges"]
ALL_COMPLETE = status_before["artifacts_complete"]

DATABASE_REQUIRED_FOR_RESUME = (
    not build_graphs
    or
    not metadata
)

PREPROCESS_REQUIRED = (
    not ALL_COMPLETE
)

if ALL_COMPLETE:
    print("\nDecision: ALL COMPLETE.")
    print("  - No database restore needed.")
    print("  - No preprocessing needed.")
    print("  - Next step: GPU Baseline Smoke Test (RUN_BASELINE_SMOKE=True).")
elif DATABASE_REQUIRED_FOR_RESUME:
    print("\nDecision: Database restore required.")
    print("  - build_graphs or metadata is incomplete.")
    print("  - PostgreSQL restore is required before preprocessing.")
    print("  - Set RESTORE_DATABASE=True to restore from Drive dump.")
elif not embed_nodes or not embed_edges:
    print("\nDecision: Restartable preprocessing (no database restore needed).")
    print("  - build_graphs and metadata are complete.")
    print("  - Word2Vec / embed_edges can resume from existing artifacts.")
    print("  - Set RUN_PREPROCESS=True to continue.")
else:
    print("\nDecision: Status unclear; manual inspection required.")


## 7. PostgreSQL archive compatibility and restore

The helper first asks installed `pg_restore` clients to list the archive. Only if none can parse it does it configure PGDG and install the validated compatible major. It never deletes cluster data. A random password is set first through the Unix socket as the `postgres` system user; command display is redacted, then TCP authentication is verified.


In [ ]:
if not RESTORE_DATABASE:
    print("RESTORE_DATABASE=False; PostgreSQL is untouched.")
else:
    from colab_postgres import restore_database
    db_name = (
        cfg.dataset.database_all_file
        if cfg.graph_construction.build_graphs.use_all_files
        else cfg.dataset.database
    )
    credentials = restore_database(
        DATABASE_DUMP, db_name, force=FORCE_DATABASE_RESTORE
    )
    os.environ["ORTHRUS_DB_HOST"] = credentials["host"]
    os.environ["ORTHRUS_DB_PORT"] = credentials["port"]
    os.environ["ORTHRUS_DB_USER"] = credentials["user"]
    os.environ["ORTHRUS_DB_PASSWORD"] = credentials["password"]

    # Rebuild cfg after environment variables are set; stale cfg is forbidden.
    cfg = fresh_preprocess_cfg()
    assert cfg.database.host == "localhost"
    assert str(cfg.database.port) == "5432"
    assert cfg.database.user == "postgres"
    from provnet_utils import init_database_connection
    cursor, connection = init_database_connection(cfg)
    try:
        cursor.execute("SELECT 1 FROM event_table LIMIT 1;")
        print("ORTHRUS database preflight: ok")
    finally:
        cursor.close()
        connection.close()


## 8. Optional bounded-memory benchmark

In [ ]:
if not RUN_PREPROCESS_BENCHMARK:
    print("RUN_PREPROCESS_BENCHMARK=False; benchmark skipped.")
else:
    benchmark_command = [
        sys.executable, str(PROJECT_ROOT / "scripts/benchmark_preprocess_memory.py"),
        "--events", "200000", "--fetch-size", "8192",
    ]
    benchmark = subprocess.run(benchmark_command, cwd=PROJECT_ROOT, check=False)
    if benchmark.returncode:
        raise RuntimeError("Preprocessing benchmark failed; formal preprocessing is blocked.")


## 9. Restartable preprocessing

In [ ]:
import subprocess
import time

cfg = fresh_preprocess_cfg()
before = read_preprocess_status(cfg)
print_status("Before:", before)

if not RUN_PREPROCESS:
    print("RUN_PREPROCESS=False; preprocessing skipped.")
else:
    # Re-read status before subprocess to detect if DB is needed
    build_graphs_before = before["build_graphs"]
    metadata_before = before["metadata"]
    
    if not build_graphs_before or not metadata_before:
        db_needed = True
        print("\nWARNING: build_graphs or metadata is incomplete.")
        print("  PostgreSQL may be required. Checking ORTHRUS_DB_* env vars...")
        db_env_vars = [k for k in os.environ if k.startswith("ORTHRUS_DB_")]
        if not db_env_vars:
            raise RuntimeError(
                "Database appears required for preprocessing, but no ORTHRUS_DB_* "
                "environment variables are set. Set RESTORE_DATABASE=True to restore "
                "from the Drive dump before preprocessing."
            )
        print(f"  Found DB env vars: {db_env_vars}")
    else:
        db_needed = False
        print("\nPreprocessing can resume without database (build_graphs + metadata complete).")
        print("  Word2Vec will load from metadata cache; embed_edges continues.")

    # Build command with --cpu flag (preprocessing does not require GPU)
    command = [
        sys.executable, str(PROJECT_ROOT / "src/orthrus.py"), DATASET,
        "--config", str(PREPROCESS_CONFIG),
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "preprocess",
        "--preprocess-substages", PREPROCESS_SUBSTAGES,
        f"--semantic_features.corpus_scope={CORPUS_SCOPE}",
        "--skip-tracing",
        "--cpu",  # Explicit CPU mode for preprocessing
    ]
    if FORCE_PREPROCESS:
        command.append("--force-preprocess")

    print(f"\nCommand: {' '.join(command)}\n")
    print("=" * 60)
    print("Preprocessing output:")
    print("=" * 60)

    # Real-time log capture to file
    log_path = ARTIFACT_ROOT / "environment" / "preprocess_latest.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)

    env = dict(os.environ)
    env["PYTHONUNBUFFERED"] = "1"

    process = subprocess.Popen(
        command,
        cwd=PROJECT_ROOT,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )

    with open(log_path, "w", encoding="utf-8") as log_file:
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
            log_file.flush()

    process.wait()
    exit_code = process.returncode

    print("=" * 60)
    print(f"Preprocessing exited with code: {exit_code}")
    print(f"Log saved to: {log_path}")

    if exit_code != 0:
        # Print last 80 lines of log
        print("\n--- Last 80 lines of log ---")
        with open(log_path, encoding="utf-8") as f:
            lines = f.readlines()
        for line in lines[-80:]:
            print(line, end="")
        raise RuntimeError(
            f"Preprocessing failed with exit code {exit_code}. "
            f"See log: {log_path}"
        )

cfg = fresh_preprocess_cfg()
after = read_preprocess_status(cfg)
print_status("After:", after)

# Verify completion
build_graphs_after = after["build_graphs"]
metadata_after = after["metadata"]
embed_nodes_after = after["embed_nodes"]
embed_edges_after = after["embed_edges"]
ALL_COMPLETE_AFTER = after["artifacts_complete"]

if not ALL_COMPLETE_AFTER:
    incomplete = []
    if not build_graphs_after: incomplete.append("build_graphs")
    if not metadata_after: incomplete.append("metadata")
    if not embed_nodes_after: incomplete.append("embed_nodes")
    if not embed_edges_after: incomplete.append("embed_edges")
    raise RuntimeError(
        f"RUN_PREPROCESS=True but ALL COMPLETE is still False. "
        f"Incomplete stages: {incomplete}. Check log: {log_path}"
    )

print("\nTHEIA preprocessing COMPLETE")
print("  ALL COMPLETE: True")

# Final artifact stats
for label, artifact_path in {
    "build_graphs": cfg.graph_construction.build_graphs._graphs_dir,
    "word2vec": cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir,
    "edge_embeddings": cfg.edge_featurization.embed_edges._edge_embeds_dir,
    "metadata": cfg._metadata_dir,
    "whole_ARTIFACT_ROOT": ARTIFACT_ROOT,
}.items():
    stats = artifact_stats(artifact_path)
    print(f"{label}: files={stats['files']}, MB={stats['MB']:.2f}, GB={stats['GB']:.3f}")


---

## 10. Baseline Smoke Test

In [ ]:
import subprocess
import sys
import yaml
from pathlib import Path

if not RUN_BASELINE_SMOKE:
    print("RUN_BASELINE_SMOKE=False，跳过 baseline smoke test。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("Baseline Smoke Test")
    print("=" * 60)

    # Smoke configuration: 1 epoch, tracing off, bounded windows
    BASE_CONFIG = PROJECT_ROOT / "config/experiments/baseline.yml"
    SMOKE_CONFIG = ARTIFACT_ROOT / "environment/smoke_baseline_1epoch.yml"

    smoke = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
    smoke.setdefault("pipeline", {})["run_tracing"] = False
    smoke.setdefault("detection", {}).setdefault("gnn_training", {})["num_epochs"] = 1
    SMOKE_CONFIG.parent.mkdir(parents=True, exist_ok=True)
    SMOKE_CONFIG.write_text(yaml.safe_dump(smoke, sort_keys=False), encoding="utf-8")

    # Build command with bounded smoke parameters
    command = [
        sys.executable,
        str(PROJECT_ROOT / "src/experiments/run_experiment.py"),
        "--dataset", DATASET,
        "--config", str(SMOKE_CONFIG),
        "--seed", str(SEEDS[0]),
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "train,test,evaluate",
    ]
    # Add bounded smoke limit if configured
    if SMOKE_MAX_WINDOWS_PER_SPLIT is not None:
        command.extend(["--max-windows-per-split", str(SMOKE_MAX_WINDOWS_PER_SPLIT)])

    print(f"Dataset:              {DATASET}")
    print(f"Seed:                {SEEDS[0]}")
    print(f"Epochs:              1")
    print(f"Max windows/split:   {SMOKE_MAX_WINDOWS_PER_SPLIT}")
    print(f"Tracing:             False")
    print(f"Smoke identity:      bounded-smoke")
    print(f"Smoke output:        {SMOKE_CONFIG}")
    print()
    print(f"Command: {' '.join(command)}")
    print()
    print("=" * 60)
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print("=" * 60)
    print("Baseline bounded smoke PASSED")
    print("=" * 60)


---

## 11. 主模型矩阵

In [ ]:
import subprocess
import sys

if not RUN_MAIN_MATRIX:
    print("RUN_MAIN_MATRIX=False，跳过主模型矩阵。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("主模型矩阵")
    print("=" * 60)

    MAIN_CONFIGS = [PROJECT_ROOT / "config/experiments/baseline.yml", PROJECT_ROOT / "config/experiments/mstc_full.yml"]

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_matrix.py"), "--datasets", DATASET, "--configs", ",".join(map(str, MAIN_CONFIGS)), "--seeds", ",".join(map(str, SEEDS)), "--artifact-root", str(ARTIFACT_ROOT)]
    result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)

    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    print("矩阵完成")
    print("=" * 60)

---

## 12. 消融与专项实验

In [ ]:
import subprocess
import sys
from pathlib import Path

if not RUN_ABLATIONS:
    print("RUN_ABLATIONS=False，跳过消融实验。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print(f"消融与专项实验: {EXPERIMENT_GROUP}")
    print("=" * 60)

    GROUPS = {
        "ablation": ["baseline.yml", "ablation_no_multiscale.yml", "ablation_no_gate.yml", "ablation_no_time.yml", "ablation_no_calibration.yml", "ablation_no_topk.yml", "mstc_full.yml"],
        "multiscale": ["multiscale_recent20.yml", "multiscale_recent24.yml", "multiscale_single_window.yml", "multiscale_equal.yml", "multiscale_gate.yml"],
        "time": ["time_type_only.yml", "time_time_only.yml", "time_joint.yml"],
        "calibration": ["calibration_max.yml", "calibration_quantile.yml", "calibration_kmeans.yml", "calibration_global_p.yml", "calibration_relation.yml", "calibration_hierarchical.yml"],
        "backbone": ["backbone_graphtransformer.yml", "backbone_graphsage_baseline.yml", "backbone_graphsage.yml", "backbone_mlp.yml"],
        "dataset_view": ["host_only.yml", "host_network_structure.yml", "host_network_full.yml"],
        "efficiency": ["baseline.yml", "efficiency_multiscale.yml", "efficiency_multiscale_time.yml", "mstc_full.yml"],
    }

    if EXPERIMENT_GROUP not in GROUPS:
        raise ValueError(f"未知组 {EXPERIMENT_GROUP!r}；可选 {sorted(GROUPS.keys())}")

    CONFIGS = [PROJECT_ROOT / "config/experiments" / name for name in GROUPS[EXPERIMENT_GROUP]]

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_matrix.py"), "--datasets", DATASET, "--configs", ",".join(map(str, CONFIGS)), "--seeds", ",".join(map(str, SEEDS)), "--artifact-root", str(ARTIFACT_ROOT)]
    result = subprocess.run(command, cwd=PROJECT_ROOT, check=False)

    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    print("消融实验完成")
    print("=" * 60)

---

## 13. Checkpoint Resume

In [ ]:
import subprocess
import sys

if not RUN_MANUAL_RESUME:
    print("RUN_MANUAL_RESUME=False，未加载任何 checkpoint。")
elif not GPU_READY:
    print("⚠️  GPU 不可用，跳过 GPU 实验。")
else:
    print("=" * 60)
    print("Checkpoint Resume")
    print("=" * 60)

    if not CHECKPOINT or not Path(CHECKPOINT).exists():
        raise FileNotFoundError(f"CHECKPOINT 不存在: {CHECKPOINT}")

    command = [sys.executable, str(PROJECT_ROOT / "src/experiments/run_experiment.py"), "--dataset", DATASET, "--config", str(RESUME_CONFIG), "--seed", str(SEEDS[0]), "--artifact-root", str(ARTIFACT_ROOT), "--stages", "train,test,evaluate", "--checkpoint", str(CHECKPOINT)]
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)
    print("Resume 完成")
    print("=" * 60)

---

## 14. 结果收集与导出

In [ ]:
import subprocess
import sys

if not RUN_COLLECT_EXPORT:
    print("RUN_COLLECT_EXPORT=False，跳过结果收集与导出。")
else:
    print("=" * 60)
    print("结果收集与导出")
    print("=" * 60)

    collect_command = [sys.executable, str(PROJECT_ROOT / "src/experiments/collect_results.py"), "--artifact-root", str(ARTIFACT_ROOT)]
    subprocess.run(collect_command, cwd=PROJECT_ROOT, check=True)

    export_command = [sys.executable, str(PROJECT_ROOT / "src/experiments/export_tables.py"), "--artifact-root", str(ARTIFACT_ROOT)]
    subprocess.run(export_command, cwd=PROJECT_ROOT, check=True)

    print("结果收集与导出完成")
    print("=" * 60)

---

## 15. 结果展示

In [ ]:
from pathlib import Path
import pandas as pd

if not RUN_DISPLAY_RESULTS:
    print("RUN_DISPLAY_RESULTS=False，跳过结果展示。")
else:
    print("=" * 60)
    print("结果展示")
    print("=" * 60)

    RESULTS_ROOT = ARTIFACT_ROOT / "results"
    table_names = ["all_runs.csv", "main_results.csv", "ablation_results.csv", "calibration_results.csv", "efficiency_results.csv"]

    for name in table_names:
        path = RESULTS_ROOT / name
        if not path.is_file():
            print(f"警告: {name} 不存在")
            continue
        df = pd.read_csv(path)
        print(f"\n--- {name} ({len(df)} rows) ---")
        display(df)

    print("=" * 60)